## Download raw files and transform them in readable CSV

In [ ]:
filename = 'ADTC_8_adz_raw_20260309'

In [44]:
from data_download import downloader
from data_decoder import decoder

downloader(filename)
decoder(filename)

Connection with geo-amberg.ch
Successful access to /45-M-02049_Opfikon_Riet/U_N20/data/ADTC_RAW
D_05_adz_raw_20260302_0606.dat successfully downloaded
D_05_adz_raw_20260302_1436.dat successfully downloaded
D_05_adz_raw_20260302_0709.dat successfully downloaded
D_05_adz_raw_20260302_2134.dat successfully downloaded
D_05_adz_raw_20260302_2051.dat successfully downloaded
D_05_adz_raw_20260302_1956.dat successfully downloaded
D_05_adz_raw_20260302_1710.dat successfully downloaded
D_05_adz_raw_20260302_2106.dat successfully downloaded
D_05_adz_raw_20260302_0936.dat successfully downloaded
D_05_adz_raw_20260302_1336.dat successfully downloaded
D_05_adz_raw_20260302_2004.dat successfully downloaded
D_05_adz_raw_20260302_0810.dat successfully downloaded
D_05_adz_raw_20260302_1610.dat successfully downloaded
D_05_adz_raw_20260302_1141.dat successfully downloaded
D_05_adz_raw_20260302_0858.dat successfully downloaded
D_05_adz_raw_20260302_1929.dat successfully downloaded
D_05_adz_raw_20260302_11

## DataFrame for the CSV (filename, timestamp)

In [45]:
import os
import pandas as pd
from pathlib import Path
from utils import date_extractor

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'RAW_DATA/{filename}'


rows = []
for file in os.listdir(path):
    file_path = os.path.join(path, file)

    with open(file_path, 'r') as f:
        content = f.read()
        micro_sec = int(content.split(';')[0][4:]) # in the raw data -> get the timestamp in ms

    time = date_extractor(micro_sec) # function defined in utils.py -> return the time in the day !

    date = os.path.basename(file_path).split('_')[4]
    yyyy = date[:4]
    mm = date[4:6]
    dd = date[6:]

    timestamp = f'{yyyy}-{mm}-{dd} {time}' # timestamp in format: YYYY-MM-DD HH:MM:SS
    timestamp = pd.to_datetime(timestamp)

    rows.append({
        'file': file.split('.')[0],
        'timestamp_raw': timestamp
    })

df_raw_data = pd.DataFrame(rows)
df_raw_data.head(5)

,file,timestamp_raw
0,D_05_adz_raw_20260302_0540,2026-03-02 05:27:48.312
1,D_05_adz_raw_20260302_0606,2026-03-02 05:57:36.573
2,D_05_adz_raw_20260302_0640,2026-03-02 06:28:13.312
3,D_05_adz_raw_20260302_0709,2026-03-02 06:57:18.096
4,D_05_adz_raw_20260302_0740,2026-03-02 07:27:28.077


## Get sensor data from the GeoVis

In [46]:
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import pandas as pd
from utils import main

env_path = 'GeoVis/DTC.env'
load_dotenv(env_path)

LOGIN = os.getenv('GEOVIS_LOGIN')
PASSWORD = os.getenv('GEOVIS_PASSWORD')


# project ID (change it for each project)
if filename.startswith('A'): # Ruschlikon
    PROJECT_ID = '1113' # Ruschlikon
elif filename.startswith('D'): # Opfkon
    PROJECT_ID = '817' # Opfikon
else:
    print('Error with the filename')

login_data = {'Login': LOGIN, 'Password': PASSWORD}
sensor_name = '_Peak'

if PROJECT_ID == "1113":
    Projekt_DB = f"{PROJECT_ID}_0"
elif PROJECT_ID == "817":
    Projekt_DB =f"{PROJECT_ID}_7"
else:
    raise ValueError("Unknown PROJECT_ID")

datum = filename.split('_')[-1]

dfs_raw_3 = main(login_data, PROJECT_ID, Projekt_DB, sensor_name, datum)

Start Calculation: 2026-03-02T00:00:00
End Calculation  : 2026-03-03T00:00:00
Datenbezug abgeschlossen.
Gefundene Sensoren: 6
Peak Detektion abgeschlossen.
2026-03-02 11:23:01.377000 -> uebersprungen: anzahl_achsen < 4
2026-03-02 15:51:13.374000 -> uebersprungen: anzahl_achsen < 4
2026-03-02 22:00:46.109000 -> uebersprungen: keine Peaks
Lokale Paar- und Sensorgeschwindigkeiten gespeichert.


### Train passage over sensor

In [47]:
from GeoVis.Preprocessing.Peak_Detektion import detect_peaks

dfs_raw = detect_peaks(dfs_raw_3)

all_peaks = []
result = []

for val in dfs_raw.values():
    peaks = val["Peaks"]

    if filename.startswith('A'): # Rueschlikon
        peaks_sensor = peaks[peaks["Sensor"].str.contains("ADTC_8")] # Rueschlikon (sensor ADTC_8)
    elif filename.startswith('D'): # Opfikon
        peaks_sensor = peaks[peaks["Sensor"].str.contains("D_05")] # Opfikon (sensor D_05)

    all_peaks.append(peaks_sensor[["Time", "Value", "Sensor"]])

peaks_df = pd.concat(all_peaks, ignore_index=True)

peaks_df["Time"] = pd.to_datetime(peaks_df["Time"])
peaks_df = peaks_df.sort_values('Time').reset_index(drop=True)

time_diff = peaks_df["Time"].diff().dt.total_seconds().fillna(0)

peaks_df["train_nr"] = (time_diff>60).cumsum()+1

for train_nr, group in peaks_df.groupby("train_nr"):
    times = group["Time"].tolist()
    
    timestamp = times[0]

    train_deltas = [
        (times[i] - times[i-1]).total_seconds()
        for i in range(1, len(times))
    ]

    result.append({
        "timestamp": timestamp,
        "timestamp_serie": times,
        "delta_t": train_deltas
    })

df_geovis_time = pd.DataFrame(result)
df_geovis_time.head(5)

Peak Detektion abgeschlossen.


,timestamp,timestamp_serie,delta_t
0,2026-03-02 05:27:57.933,"[2026-03-02 05:27:57.933000, 2026-03-02 05:27:...","[0.499, 0.489, 0.16, 0.519, 0.44, 0.38, 0.02, ..."
1,2026-03-02 05:57:46.193,"[2026-03-02 05:57:46.193000, 2026-03-02 05:57:...","[0.869, 0.13, 0.389, 0.399, 0.549, 0.18, 0.17,..."
2,2026-03-02 06:28:25.129,"[2026-03-02 06:28:25.129000, 2026-03-02 06:28:...","[0.909, 0.969, 0.47, 0.24, 1.008, 0.479, 0.26,..."
3,2026-03-02 06:57:29.703,"[2026-03-02 06:57:29.703000, 2026-03-02 06:57:...","[0.919, 1.009, 0.41, 0.29, 0.21, 0.829, 0.479,..."
4,2026-03-02 07:27:39.795,"[2026-03-02 07:27:39.795000, 2026-03-02 07:27:...","[0.709, 0.18, 0.15, 0.789, 0.42, 0.27, 0.2, 0...."


### Velocity over sensor

In [48]:
# use dfs_raw_3 -> results of the function 'main()'

rows = []

for ts, data in dfs_raw_3.items():
    sensor = data.get('geschw_pro_achse_sensor_local')

    if sensor is None:
        continue

    # replace 5 with the sensor you want to use to extract data
    if 5 in sensor:
        rows.append({
            'timestamp': ts,
            'velocity': sensor[5]
        })

df_geovis_velocity = pd.DataFrame(rows)
df_geovis_velocity.head(5)

,timestamp,velocity
0,2026-03-02 05:27:56.366,"[9.284369114877588, 9.319470699432891, 9.57281..."
1,2026-03-02 05:57:40.321,"[13.80952380952381, 13.80952380952381, 13.6565..."
2,2026-03-02 06:02:02.248,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
3,2026-03-02 06:28:24.390,"[10.313807531380753, 10.1440329218107, 10.6941..."
4,2026-03-02 06:57:28.855,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."


### Merge both GeoVis DataFrame and compute traing length

In [49]:
df_geovis = pd.merge_asof(
    df_geovis_time.sort_values('timestamp'),
    df_geovis_velocity.sort_values('timestamp'),
    left_on='timestamp',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

# add the length between every axis, computed using d=v*t
df_geovis['length'] = df_geovis.apply(
    lambda row: [
        a * b for a, b in zip(row["delta_t"], row["velocity"])
    ] if isinstance(row["delta_t"], list)
    and isinstance(row["velocity"], list)
    and len(row["delta_t"]) > 1
    and len(row["velocity"]) > 1
    else [],
    axis=1
)

# add the total train length, by summin the values in the length list
df_geovis['total_length'] = df_geovis['length'].apply(sum)
df_geovis = df_geovis[df_geovis['total_length'] != 0] # remove all entries where the total length is 0

df_geovis.head(5)

,timestamp,timestamp_serie,delta_t,velocity,length,total_length
0,2026-03-02 05:27:57.933,"[2026-03-02 05:27:57.933000, 2026-03-02 05:27:...","[0.499, 0.489, 0.16, 0.519, 0.44, 0.38, 0.02, ...","[9.284369114877588, 9.319470699432891, 9.57281...","[4.632900188323917, 4.557221172022683, 1.53165...",153.582983
1,2026-03-02 05:57:46.193,"[2026-03-02 05:57:46.193000, 2026-03-02 05:57:...","[0.869, 0.13, 0.389, 0.399, 0.549, 0.18, 0.17,...","[13.80952380952381, 13.80952380952381, 13.6565...","[12.00047619047619, 1.7952380952380953, 5.3123...",104.919584
2,2026-03-02 06:28:25.129,"[2026-03-02 06:28:25.129000, 2026-03-02 06:28:...","[0.909, 0.969, 0.47, 0.24, 1.008, 0.479, 0.26,...","[10.313807531380753, 10.1440329218107, 10.6941...","[9.375251046025106, 9.829567901234569, 5.02624...",181.152769
3,2026-03-02 06:57:29.703,"[2026-03-02 06:57:29.703000, 2026-03-02 06:57:...","[0.919, 1.009, 0.41, 0.29, 0.21, 0.829, 0.479,...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",NaN
4,2026-03-02 07:27:39.795,"[2026-03-02 07:27:39.795000, 2026-03-02 07:27:...","[0.709, 0.18, 0.15, 0.789, 0.42, 0.27, 0.2, 0....","[10.378947368421052, 10.647948164146868, 11.00...","[7.358673684210525, 1.9166306695464361, 1.6506...",206.448423


## Merge DataFrame from the CSV and DataFrame from GeoVIS

In [50]:
df_merged = pd.merge_asof(
    df_geovis.sort_values('timestamp'),
    df_raw_data.sort_values('timestamp_raw'),
    left_on='timestamp',
    right_on='timestamp_raw',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

df_merged = df_merged.dropna(subset=['file', 'timestamp_raw'])

cols = ['file', 'timestamp_raw'] + [
    c for c in df_merged.columns
    if c not in ['file', 'timestamp_raw']
]

df_merged = df_merged[cols]

df_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length
0,D_05_adz_raw_20260302_0540,2026-03-02 05:27:48.312,2026-03-02 05:27:57.933,"[2026-03-02 05:27:57.933000, 2026-03-02 05:27:...","[0.499, 0.489, 0.16, 0.519, 0.44, 0.38, 0.02, ...","[9.284369114877588, 9.319470699432891, 9.57281...","[4.632900188323917, 4.557221172022683, 1.53165...",153.582983
1,D_05_adz_raw_20260302_0606,2026-03-02 05:57:36.573,2026-03-02 05:57:46.193,"[2026-03-02 05:57:46.193000, 2026-03-02 05:57:...","[0.869, 0.13, 0.389, 0.399, 0.549, 0.18, 0.17,...","[13.80952380952381, 13.80952380952381, 13.6565...","[12.00047619047619, 1.7952380952380953, 5.3123...",104.919584
2,D_05_adz_raw_20260302_0640,2026-03-02 06:28:13.312,2026-03-02 06:28:25.129,"[2026-03-02 06:28:25.129000, 2026-03-02 06:28:...","[0.909, 0.969, 0.47, 0.24, 1.008, 0.479, 0.26,...","[10.313807531380753, 10.1440329218107, 10.6941...","[9.375251046025106, 9.829567901234569, 5.02624...",181.152769
3,D_05_adz_raw_20260302_0709,2026-03-02 06:57:18.096,2026-03-02 06:57:29.703,"[2026-03-02 06:57:29.703000, 2026-03-02 06:57:...","[0.919, 1.009, 0.41, 0.29, 0.21, 0.829, 0.479,...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",NaN
4,D_05_adz_raw_20260302_0740,2026-03-02 07:27:28.077,2026-03-02 07:27:39.795,"[2026-03-02 07:27:39.795000, 2026-03-02 07:27:...","[0.709, 0.18, 0.15, 0.789, 0.42, 0.27, 0.2, 0....","[10.378947368421052, 10.647948164146868, 11.00...","[7.358673684210525, 1.9166306695464361, 1.6506...",206.448423


## Cut the CSV (First-Last Peak)

In [51]:
import numpy as np
from scipy.signal import savgol_filter
from peaks import peakFind, firstLastPeak

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'CSV_DATA/{filename}_csv'
dst = root / f'CSV_DATA/{filename}_cut'
os.makedirs(dst, exist_ok=True)

for file in os.listdir(src):
    file_path = os.path.join(src, file)
    df = pd.read_csv(file_path)

    time = df['Time [s]'].values
    signal = df['Distance[mm]'].values

    fs = 1/np.mean(np.diff(time))
    signal_smooth = savgol_filter(signal, 101, 3)

    # peak detection
    peakValues, peakTimes = peakFind(time, signal_smooth, fs)
    firstValue, firstTime, lastValue, lastTime = firstLastPeak(peakValues, peakTimes)

    # add first peak in df_merged
    df_merged['first_peak_raw'] = firstTime
    
    name = file.split('.')[0]

    # cut the csv
    subset = df_merged.loc[df_merged['file'] == name, 'velocity']
    if subset.empty:
        print(f'Skipping {file}')
        continue

    v = subset.iloc[0]
    velocity_start = v[0] if isinstance(v, list) else v
    velocity_end = v[-1] if isinstance(v, list) else v

    start_sec_m = 1/velocity_start
    end_sec_m = 1/velocity_end

    start = max(0, firstTime - start_sec_m)
    end = lastTime + end_sec_m
    mask = (df["Time [s]"] >= start) & (df["Time [s]"] <= end)
    df_cut = df.loc[mask].copy()

    if df_cut.empty:
        print(f"Empty cut for file: {file}")
        continue

    out_path = os.path.join(dst, file.replace(".csv", "_cut.csv"))
    df_cut.to_csv(out_path, index=False)

df_merged.head(5)

Empty cut for file: D_05_adz_raw_20260302_0709.csv
Empty cut for file: D_05_adz_raw_20260302_0858.csv
Empty cut for file: D_05_adz_raw_20260302_0912.csv
Empty cut for file: D_05_adz_raw_20260302_1017.csv
Empty cut for file: D_05_adz_raw_20260302_1042.csv
Empty cut for file: D_05_adz_raw_20260302_1102.csv
Empty cut for file: D_05_adz_raw_20260302_1115.csv
Skipping D_05_adz_raw_20260302_1129.csv
Empty cut for file: D_05_adz_raw_20260302_1141.csv
Empty cut for file: D_05_adz_raw_20260302_1235.csv
Empty cut for file: D_05_adz_raw_20260302_1250.csv
Empty cut for file: D_05_adz_raw_20260302_1348.csv
Empty cut for file: D_05_adz_raw_20260302_1436.csv
Empty cut for file: D_05_adz_raw_20260302_1506.csv
Empty cut for file: D_05_adz_raw_20260302_1534.csv
Skipping D_05_adz_raw_20260302_1557.csv
Skipping D_05_adz_raw_20260302_1610.csv
Empty cut for file: D_05_adz_raw_20260302_1623.csv
Empty cut for file: D_05_adz_raw_20260302_1839.csv
Empty cut for file: D_05_adz_raw_20260302_1929.csv
Skipping D_05

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw
0,D_05_adz_raw_20260302_0540,2026-03-02 05:27:48.312,2026-03-02 05:27:57.933,"[2026-03-02 05:27:57.933000, 2026-03-02 05:27:...","[0.499, 0.489, 0.16, 0.519, 0.44, 0.38, 0.02, ...","[9.284369114877588, 9.319470699432891, 9.57281...","[4.632900188323917, 4.557221172022683, 1.53165...",153.582983,10.755039
1,D_05_adz_raw_20260302_0606,2026-03-02 05:57:36.573,2026-03-02 05:57:46.193,"[2026-03-02 05:57:46.193000, 2026-03-02 05:57:...","[0.869, 0.13, 0.389, 0.399, 0.549, 0.18, 0.17,...","[13.80952380952381, 13.80952380952381, 13.6565...","[12.00047619047619, 1.7952380952380953, 5.3123...",104.919584,10.755039
2,D_05_adz_raw_20260302_0640,2026-03-02 06:28:13.312,2026-03-02 06:28:25.129,"[2026-03-02 06:28:25.129000, 2026-03-02 06:28:...","[0.909, 0.969, 0.47, 0.24, 1.008, 0.479, 0.26,...","[10.313807531380753, 10.1440329218107, 10.6941...","[9.375251046025106, 9.829567901234569, 5.02624...",181.152769,10.755039
3,D_05_adz_raw_20260302_0709,2026-03-02 06:57:18.096,2026-03-02 06:57:29.703,"[2026-03-02 06:57:29.703000, 2026-03-02 06:57:...","[0.919, 1.009, 0.41, 0.29, 0.21, 0.829, 0.479,...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",NaN,10.755039
4,D_05_adz_raw_20260302_0740,2026-03-02 07:27:28.077,2026-03-02 07:27:39.795,"[2026-03-02 07:27:39.795000, 2026-03-02 07:27:...","[0.709, 0.18, 0.15, 0.789, 0.42, 0.27, 0.2, 0....","[10.378947368421052, 10.647948164146868, 11.00...","[7.358673684210525, 1.9166306695464361, 1.6506...",206.448423,10.755039


## Add length to the CSV

In [52]:
for file in os.listdir(dst):

    file_path = os.path.join(dst, file)
    df_cut = pd.read_csv(file_path)

    name = file.split("_cut")[0]

    # get merged row
    row = df_merged.loc[df_merged["file"] == name]

    if row.empty:
        print(f"Missing in df_merged: {name}")
        continue

    row = row.iloc[0]


    times = pd.to_datetime(row["timestamp_serie"])
    peak_times = pd.to_datetime(times).values.astype("datetime64[ns]")

    t0 = peak_times[0]
    t_peaks = (peak_times - t0) / np.timedelta64(1, "s")

    cum_length = np.concatenate([[0], np.cumsum(row["length"])])


    # first peak reference (from cut file)
    time_shift = df_cut["Time [s]"].values
    time_shift = time_shift - time_shift[0]  # ensure starts at 0

    # interpolate length
    min_len = min(len(t_peaks), len(cum_length))

    t_peaks = t_peaks[:min_len]
    cum_length = cum_length[:min_len]

    df_cut["Length [m]"] = np.interp(
        time_shift,
        t_peaks,
        cum_length
    )

    # save file
    df_cut.to_csv(file_path, index=False)

In [53]:
df_merged.to_csv(f'DataFrame/df_merged_{filename}.csv', index=False)